In [ ]:
import os, json, sys
from pathlib import Path
import torch
import numpy as np
from PIL import Image
from IPython.display import display

In [ ]:
DATA_SET_NAME = "Anti-UAV-RGBT"
DATA_SET_NAME = "Anti-UAV410"
DATA_SET_NAME = "CST-AntiUAV"

ROOT_DIR = Path.cwd().parent#
print(f"Root-Verzeichnis: {ROOT_DIR}")
DATA_DIR = ROOT_DIR / "data"
OUTPUT_DIR = ROOT_DIR / "output"

DATASET_DIR  = DATA_DIR / DATA_SET_NAME
TRAINING_DIR = DATASET_DIR / "train"

Root-Verzeichnis: /home/jessh/Study/Bachelor-Thesis


In [ ]:
SRC_DIR = ROOT_DIR / "src"
if str(SRC_DIR) not in sys.path:
    sys.path.append(str(SRC_DIR))
'''
try:
    import data_handler
    print("Erfolgreich importiert:", data_handler)
except ImportError as e:
    print(f"Fehler beim Importieren: {e}")
'''

Erfolgreich importiert: <module 'data_handler' from '/home/jessh/Study/Bachelor-Thesis/src/data_handler/__init__.py'>


In [ ]:
def get_all_dir(root_dir, file_endings=['.png', '.jpg', '.jpeg']):
    video_pathes = []
    for dir in os.listdir(root_dir):
        dir = os.path.join(root_dir, dir)
        if os.path.isdir(dir):
            video_pathes.extend(get_all_dir(dir))
        if dir[-4:] in file_endings:
            video_pathes.append(Path(dir))
    return video_pathes

In [ ]:
def get_all_dir_names(root_dir):
    root = Path(root_dir)
    dir_paths = [str(p) for p in root.rglob("*") if p.is_dir()]
    return dir_paths

In [ ]:
def show_img(img):
    img_local = img
    # Falls es ein PyTorch-Tensor ist, in ein NumPy-Array konvertieren
    if isinstance(img, torch.Tensor):
        img_local = img.detach().cpu()
        
        # Falls Shape [C, H, W] ist (z.B. [1, H, W] oder [3, H, W])
        if img_local.ndim == 3:
            if img_local.shape[0] == 1:
                img_local = img_local.squeeze(0)  # [1, H, W] -> [H, W]
            else:
                img_local = img_local.permute(1, 2, 0)  # [3, H, W] -> [H, W, 3]
                
        # Skalierung von [0.0, 1.0] zurück auf [0, 255] und als uint8 casten
        img_local = (img_local * 255).clamp(0, 255).numpy().astype(np.uint8)
        
    display(Image.fromarray(img_local))

In [ ]:
# ---------------------------------------------------------
# Hilfsfunktionen zur Bounding-Box / Annotationen-Suche
# ---------------------------------------------------------
def load_annotations(seq_dir):
    """
    Sucht nach Label-Dateien (JSON oder TXT/YOLO) im Sequenz-Ordner.
    Unterstützt gängige Anti-UAV Formate (IR_label.json, txt, etc.)
    """
    seq_path = Path(seq_dir)
    json_files = list(seq_path.glob("*.json"))
    txt_files = list(seq_path.glob("*.txt"))
    
    bboxes = [] # Format: [x, y, w, h] (in Pixeln)
    
    if json_files:
        for jf in json_files:
            try:
                with open(jf, 'r') as f:
                    data = json.load(f)
                    # Falls COCO / Anti-UAV JSON Struktur
                    if isinstance(data, dict):
                        if 'gt_rect' in data:
                            return data['gt_rect'] # Oft [[x,y,w,h], ...]
                        elif 'gt' in data:  # For CST-AntiUAV
                            return data['gt'] 
                        elif 'exist' in data and 'gt_rect' in data:
                            return [rect for exist, rect in zip(data['exist'], data['gt_rect']) if exist]
            except Exception:
                pass
                
    elif txt_files:
        # Falls YOLO / TXT Format vorhanden
        for tf in txt_files:
            if tf.name in ["labels.txt", "IR_label.txt"] or len(txt_files) == 1:
                try:
                    with open(tf, 'r') as f:
                        lines = f.readlines()
                        for l in lines:
                            parts = [float(x) for x in l.strip().split()]
                            if len(parts) >= 4:
                                bboxes.append(parts[-4:]) # nimmt x,y,w,h
                except Exception:
                    pass
    return bboxes
